# Climate scenarios

In [ ]:
# general
from pathlib import Path
import pandas as pd
import yaml 
import os

from src.utils import load_config
cfg = load_config()

## Data

In [ ]:
# temperature levels for CMIP6 models in Hauser et al 2022
temperature_levels = pd.read_csv("data/climate/cmip6_models_temp_levels_1850_1900.csv")
temperature_levels = temperature_levels.map(lambda x: x.strip() if isinstance(x, str) else x)

# transient climate response (TCR) and equilibrium climate sensitivity (ECS) in Hausfather et al 2022
models_selected    = pd.read_csv("data/climate/cmip6_models_TCR_ECS.csv")
models_selected    = models_selected[models_selected["TCR Screen (likely) 1.4-2.2º"] == "Y"]

# available bias-corrected models in Gergel et al 2024
gcm_metadata   = pd.read_csv("data/climate/cmip6_models_GDPCIR_metadata.csv")

## Filtering and aggregation based on warming levels

In [ ]:
# no bias corrected data for this scenario
temperature_levels = temperature_levels[temperature_levels.exp != "ssp119"]

# target period
temperature_levels = temperature_levels[temperature_levels.period == '2081-2100']

# filter by available gcms (CMIP6 bias corrected data)
temperature_levels = temperature_levels.merge(models_selected['model'],  
                                               on=['model'],  how='inner')

# filter by TCR Screen (likely) 1.4-2.2ºC
temperature_levels = temperature_levels.merge(gcm_metadata[['model', 'ensemble']],  
                                               on=['model', 'ensemble'],  how='inner')

In [ ]:
# Define the filtering conditions for each group
tolerance = 0.25
group_15C = temperature_levels[(temperature_levels["mean"] > 1.5-tolerance) & (temperature_levels["mean"] < 1.5+tolerance)].copy()
group_28C = temperature_levels[(temperature_levels["mean"] > 2.8-tolerance) & (temperature_levels["mean"] < 2.8+tolerance)].copy()
group_40C = temperature_levels[(temperature_levels["mean"] > 4.0-tolerance) & (temperature_levels["mean"] < 4.0+tolerance)].copy()

# Assign labels to each group 
group_15C.loc[:, "group"] = "15C"
group_28C.loc[:, "group"] = "28C"
group_40C.loc[:, "group"] = "40C"

temperature_levels = pd.concat([group_15C, group_28C, group_40C])
temperature_levels.to_csv("data/climate/cmip6_models_selection.csv", index=False)